In [ ]:
%load_ext autoreload
%autoreload 2

from tensorboard.backend.event_processing import event_accumulator
import os
import scipy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import copy
from tqdm import tqdm
import csv
from scipy.special import softmax
device = torch.device('cuda:0')
import random

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename='train_log.log', level=logging.INFO)

from Dataset import ImportanceDataset, RealImportanceDataset, RealPredictionDataset, XBoxDatasetSimulation, Axios_ipsosdataset, HouseholdPulse_dataset
from GAN import GAN, WGAN_GP
from Discriminator import DataDiscriminator, DeepSetCritic
from util import set_seed
import itertools
import pandas as pd
#optimize over w directly, dont use any theta shenannigans. 
from sklearn import preprocessing
import pytorch_warmup as warmup
from DataProcessing import *
import pickle

In [ ]:
#d4p data processing

df = pd.read_stata("./data/progress_data/dfp_covid_tracking_poll.dta")
df = df[df['wave'] == 25]
columns_to_keep = []
columns_to_keep.append('nationalweight')
columns_to_keep.append('gender')
columns_to_keep.append('ethnicity')
columns_to_keep.append('education')
columns_to_keep.append('age')
columns_to_keep.append('region')
columns_to_keep.append('hhi') #household income
columns_to_keep.append('vax')

df = df[columns_to_keep].dropna(subset=columns_to_keep)


vax_binary = df['vax'].str.startswith('Yes,').astype(int)
weighted_avg = (vax_binary * df['nationalweight']).sum() / df['nationalweight'].sum()

#d4p procressing
df = pd.read_stata("./data/progress_data/dfp_covid_tracking_poll.dta")
df = df[df['wave'] == 25]

columns_to_keep = []
columns_to_keep.append('gender')
columns_to_keep.append('ethnicity')
columns_to_keep.append('education')
columns_to_keep.append('age')
columns_to_keep.append('region')
columns_to_keep.append('hhi') #household income
columns_to_keep.append('vax')

df = df[columns_to_keep].dropna(subset=columns_to_keep)

#d4p processing 2
def recode_gender(value):
    if value == 'Male':
        return 1
    elif value == 'Female':
        return 2
    else:
        return None
df['gender'] = df['gender'].apply(recode_gender)

def recode_census_age(value):
    if value >= 85: 
        return 5
    elif value >= 65:
        return 4
    elif value >= 50:
        return 3
    elif value >= 35:
        return 2
    elif value >= 18:
        return 1
df['age'] = df['age'].apply(recode_census_age)

def recode_census_region(value):
        if value == 1:
            return 1  # NorthEast
        elif value == 2:
            return 3  # MidWest
        elif value == 3:
            return 2  # South
        elif value == 4:
            return 4  # West
        else:
            return None  # Handle unexpected values
# Apply the recoding function to the region column in the census data
df['region'] = df['region'].apply(recode_census_region)

educ_mapping = {
            1: 1,  # Less than high school
            2: 2,  # High school graduate
            3: 3,  # Some college
            4: 3,  # Some college
            5: 3,  # Some college
            6: 4, # Bachelor's degree
            7: 5,  # Graduate degree
            8: 5,
            -3105: None,
        }
df['education'] = df['education'].map(educ_mapping)

race_map = {
        1: 1,  # White -> White, Alone
        2: 2,  # Black/African American -> Black, Alone
        3: 4,  # American Indian or Alaska Native -> Any other race alone, or race in combination
        5: 3,  # Chinese -> Asian, Alone
        7: 3,  # Japanese -> Asian, Alone
        4: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        6: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        8: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        9: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        10: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        11: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        12: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        13: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        14: 3,  # Other Asian or Pacific Islander -> Asian, Alone
        15: None,  # Other race, nec -> Any other race alone, or race in combination
        16: None,
    }
df['ethnicity'] = df['ethnicity'].map(race_map)

def map_income(value):
    if value == -3015:
        return None  # Question seen but not selected
    elif value <= 3:
        return 1  # Less than $25,000
    elif 4 <= value < 6:
        return 2  # $25,000 - $34,999
    elif 6 <= value < 9:
        return 3  # $35,000 - $49,999
    elif 9 <= value < 14:
        return 4  # $50,000 - $74,999
    elif 14 <= value < 19:
        return 5  # $75,000 - $99,999
    elif 19 <= value < 21:
        return 6  # $100,000 - $149,999
    elif 21 <= value < 23:
        return 7  # $150,000 - $199,999
    else:
        return 8  # $200,000 and above

df['hhi'] = df['hhi'].apply(map_income)

df = df.rename(columns={'gender': 'SEX'})
df = df.rename(columns={'education': 'EDUC'})
df = df.rename(columns={'region': 'REGION'})
df = df.rename(columns={'hhi': 'INCTOT'})
df = df.rename(columns={'ethnicity': 'RACE'})
df = df.rename(columns={'age': 'AGE'})

vax_binary = df['vax'].str.startswith('Yes,').astype(int)
df.drop('vax',axis=1)
df['vax'] = vax_binary

cols = list(df.columns)
cols.insert(0, cols.pop(cols.index('vax')))
df = df[cols]

print(df)

week='25'
df.to_csv('./data/progress_data/week'+week+'_cleaned.csv', index=False)

In [ ]:
#d4p processing continued 
from Dataset import D4P_dataset
EPOCHS = 100
DISC_LR = 1e-5
GT_LIMIT = 100 #25000
BIAS_LIMIT = 100 #1000
BATCH_SIZE =16
SUBSET_SIZE = 64
week='25'

seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)

D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets

d = D4P_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
            bias_path = './data/progress_data/week'+week+'_cleaned.csv',
            rngs=rngs,
            device=device,
            gt_limit = GT_LIMIT,
            )

In [ ]:
#analyzing the groupings of variables
#plotting events from saved run logs
# Path to the directory where SummaryWriter saved logs
import re
import ast
file_paths = ["runs_household_1_8_vars/"]

def isolate_variable_names(file_name):
    match = re.search(r"columns_to_keep:(\([^\)]*\))\|\|seed", file_name)
    if match:
        test = ast.literal_eval((match.group(1)))
        return test
    return None
def print_stats(runs_path):
    run_folders = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_vac_predictions = []
    all_l2_demographics = []
    all_names = []
    for i, event_file in enumerate(event_files):
        # Load event accumulator
        var_name_tuple = isolate_variable_names(event_file)
        if len(var_name_tuple) != num_vars:
            continue
        all_names.append(var_name_tuple)
        ea = event_accumulator.EventAccumulator(event_file)
        ea.Reload()
        #scalar_tags = ea.Tags().get('scalars', [])
        # List available tags (scalars, histograms, images, etc.)
        # Read scalar values (e.g., 'loss', 'accuracy')
        try:
            #demo_events = ea.Scalars("JS Divergence")
            demo_events = ea.Scalars("l2 norm demo diff")
            prediction_events = ea.Scalars("Vaccine prediction total") 
        except:
            print("error: ", event_file)
            continue
        #summ = 0
        #for iter, event in enumerate(prediction_events):
        #    print(event, l2_events[iter])
        cur_vac_pred = []
        cur_l2_demo = []
        for event in demo_events:
            cur_l2_demo.append(event.value)
        for event in prediction_events:
            cur_vac_pred.append(event.value)
        all_l2_demographics.append(cur_l2_demo)
        all_vac_predictions.append(cur_vac_pred)

    all_l2_demographics = np.array(all_l2_demographics)
    all_vac_predictions = np.array(all_vac_predictions)

    starting_prediction = []
    min_prediction = []
    for row in range(all_vac_predictions.shape[0]):
        starting_prediction.append(all_vac_predictions[row,0])
        min_prediction.append(np.min(all_vac_predictions[row,:]))

    combined = list(zip(all_names, starting_prediction, min_prediction))

    # Sort by the first element of each tuple (i.e., values from A)
    combined.sort(key=lambda x: x[2])

    # Unzip back into separate lists
    names_sorted, starting_sorted, min_sorted = zip(*combined)

    # Convert back to lists if needed
    names_sorted = list(names_sorted)
    starting_sorted = list(starting_sorted)
    min_sorted = list(min_sorted)
    
    results_dir = {}
    for ii, name_tup in enumerate(names_sorted):
        for nt in name_tup:
            if nt not in results_dir:
                results_dir[nt] = [min_sorted[ii]]
            else:
                results_dir[nt].append(min_sorted[ii])
    print("Mean min: ", np.mean(min_sorted))
    print("")
    print("Lower than mean:")
    for var in results_dir:
        if np.mean(results_dir[var]) < np.mean(min_sorted):
            print(var, np.mean(results_dir[var]))
    print("")

    print("Higher than mean:")
    for var in results_dir:
        if np.mean(results_dir[var]) > np.mean(min_sorted):
            print(var, np.mean(results_dir[var]))

    print("")

    x = 5
    for ii in range(x):
        print(names_sorted[ii], min_sorted[ii])

for runs_path in file_paths:
    print_stats(runs_path)

In [ ]:
#testing dataset creation
EPOCHS = 100
DISC_LR = 1e-5
GT_LIMIT = 100 #25000
BIAS_LIMIT = 100 #1000
BATCH_SIZE =16
SUBSET_SIZE = 64
week='29'

seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)

D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets
d = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                                    bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                                    rngs=rngs,
                                    device=device,
                                    gt_limit = GT_LIMIT,
                                    bias_limit = BIAS_LIMIT, 
                                    )

In [ ]:
#attempting to find trends in the randomness
#plotting events from saved run logs
# Path to the directory where SummaryWriter saved logs
file_paths = ["runs/"]

def print_stats(runs_path):
    run_folders = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_vac_predictions = []
    all_l2_demographics = []
    for i, event_file in enumerate(event_files):
        # Load event accumulator
        ea = event_accumulator.EventAccumulator(event_file)
        ea.Reload()

        # List available tags (scalars, histograms, images, etc.)
        # Read scalar values (e.g., 'loss', 'accuracy')
        prediction_events = ea.Scalars("Vaccine prediction")
        l2_events = ea.Scalars("L2 Demographics")
        #summ = 0
        #for iter, event in enumerate(prediction_events):
        #    print(event, l2_events[iter])
        cur_vac_pred = []
        cur_l2_demo = []
        for event in prediction_events:
            cur_vac_pred.append(event.value)
        for ievent in l2_events:
            cur_l2_demo.append(ievent.value)
        all_vac_predictions.append(cur_vac_pred)
        all_l2_demographics.append(cur_l2_demo)

    
    all_vac_predictions = np.array(all_vac_predictions)
    all_l2_demographics = np.array(all_l2_demographics)

    r_vales = []
    for i in range(1,70):
        start_point = 80
        end_point = np.shape(all_vac_predictions)[1]-i
        #ending points
        ending_vac_pred = all_vac_predictions[:,end_point]
        ending_l2_demopgrahics = all_l2_demographics[:,end_point]

        ending_vac_pred=np.delete(ending_vac_pred, ending_l2_demopgrahics.argmax())
        ending_l2_demopgrahics=np.delete(ending_l2_demopgrahics, ending_l2_demopgrahics.argmax())

        #plt.scatter(x=ending_l2_demopgrahics, y=ending_vac_pred)
        #plt.xlabel("Ending Demographic L2")
        #plt.ylabel("Ending Vac Prediction")
        #plt.show()
        slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(ending_l2_demopgrahics, ending_vac_pred)
        r_vales.append(r_value)
        #print(r_value)
        #print("R value: ", r_value)
        #print("---------------------------------")
        #average from starting point
        #avg_vac_pred = np.mean(all_vac_predictions[:,start_point:],axis=1)
        #avg_l2_demo = np.mean(all_l2_demographics[:,start_point:],axis=1)
        #print(avg_vac_pred)
        #plt.scatter(x=avg_l2_demo, y=avg_vac_pred)
        #plt.xlabel("avg Demographic L2")
        #plt.ylabel("avg Vac Prediction")
        #slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(avg_vac_pred, avg_l2_demo)
        #print("R value: ", r_value)

    plt.plot(r_vales)
for runs_path in file_paths:
    print_stats(runs_path)

In [ ]:
from HouseholdCensusDataProcessing import * 
import random
#making aggregated HHP
aggregated = load_evenly_sampled_csv_rows("./data/censusHouseholdPulse_data/raw_survey/", K=2500, target_var = 'HLTHINS1')
aggregated_cleaned, _ = recoding_survey_and_census_data(aggregated, None)
aggregated_cleaned.to_csv('./data/censusHouseholdPulse_data/cleaned/pulse_weekALL_cleaned.csv',index=False)

In [ ]:
#entropy vs prediction power visualization
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
from itertools import combinations

#analyzing consistency over different types of randomness
#plotting events from saved run logs
from tensorboard.backend.event_processing import event_accumulator
import os
# Path to the directory where SummaryWriter saved logs

def plot_2d_runs(data):
    # Compute mean and standard deviation across experiments
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)  # Or use std / sqrt(n) for standard error

    # Time steps (x-axis)
    time_steps = np.arange(data.shape[1])

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(time_steps, mean, label='Mean Time Series')
    plt.fill_between(time_steps, mean - std, mean + std, alpha=0.3, label='±1 Std Dev')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.title('Average Time Series with Error Bands')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def load_runs_as_numpy(runs_path, var_names, filter_by, num_runs=9):
    run_folders = []
    count = 0
    for name in os.listdir(runs_path):
        valid_file = True
        for fb in filter_by:
            if fb not in name:
                valid_file = False
                break
        if not valid_file:
            continue
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))
            count += 1
        if count >= num_runs:
            break

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))
    all_data = {}
    for i_name, vname in enumerate(var_names):
        all_data[vname] = []
        for i, event_file in enumerate(event_files):
            # Load event accumulator
            ea = event_accumulator.EventAccumulator(event_file)
            ea.Reload()

            # List available tags (scalars, histograms, images, etc.)
            # Read scalar values (e.g., 'loss', 'accuracy')
            if not vname in ea.Tags()["scalars"]:
                continue
            prediction_events = ea.Scalars(vname)
            #summ = 0
            #for iter, event in enumerate(prediction_events):
            #    print(event, l2_events[iter])
            cur_pred = []
            for event in prediction_events:
                cur_pred.append(event.value)
            all_data[vname].append(cur_pred)
        all_data[vname] = np.array(all_data[vname])
    return all_data, event_files

def load_all_as_dict(runs_path, var_names, filter_by):
    run_folders = []
    count = 0
    all_names = []
    for name in os.listdir(runs_path):
        if os.path.isdir(os.path.join(runs_path,name)):
            run_folders.append(os.path.join(runs_path,name))
            all_names.append(name)
            count += 1

    event_files = []
    for run_folder in run_folders:
        for name in os.listdir(run_folder):
            if os.path.isdir(os.path.join(run_folder,name)):
                continue
            else:
                event_files.append(os.path.join(run_folder,name))

    all_data = {}
    for i_name, vname in enumerate(var_names):
        all_data[vname] = []
        for i, event_file in enumerate(event_files):
            # Load event accumulator
            ea = event_accumulator.EventAccumulator(event_file)
            ea.Reload()

            # List available tags (scalars, histograms, images, etc.)
            # Read scalar values (e.g., 'loss', 'accuracy')
            prediction_events = ea.Scalars(vname)
            #summ = 0
            #for iter, event in enumerate(prediction_events):
            #    print(event, l2_events[iter])
            cur_pred = []
            for event in prediction_events:
                cur_pred.append(event.value)
            all_data[vname].append(cur_pred)
        all_data[vname] = np.array(all_data[vname])
    return all_data, all_names

def print_stats(runs_path):
    all_data = load_runs_as_numpy(runs_path, ['Vaccine prediction', 'l2 norm demo diff'])
    all_vac_predictions = all_data['Vaccine prediction']
    all_l2_demographics = all_data['l2 norm demo diff']
    vac_stds = np.std(all_vac_predictions,axis=0)
    l2_stds = np.std(all_l2_demographics,axis=0)
    print("Consistency of Vac | consistency of L2 (30 trials)")
    print(np.mean(vac_stds),np.mean(l2_stds))
    print("")

    print("Lowest point of vacc variance")
    print(np.argmin(vac_stds[25:]), np.min(vac_stds[25:]), np.mean(all_vac_predictions[:,np.argmin(vac_stds[25:])]))

    #plot_2d_runs(all_vac_predictions[:,25:])

def find_peaks_troughs(runs_path, filter_by):
    sigma = 25  # smoothing level
    mr = 30
    out = load_runs_as_numpy(runs_path,  
                             ['Vaccine prediction total', 'GenLoss', 'Gen Entropy'],
                             filter_by=filter_by)
    vpt = out['Vaccine prediction total']
    gloss = out['GenLoss']
    entropy = out['Gen Entropy']
    for i, y in enumerate(gloss):
        # Smooth the signal
        smoothed = gaussian_filter1d(y, sigma=sigma)
            # Find peaks and troughs
        peaks, _ = find_peaks(smoothed)
        troughs, _ = find_peaks(-smoothed)

        converted_peaks = [int(p/2) for p in peaks]
        converted_troughs = [int(t/2) for t in troughs]
        #print(vpt.shape, converted_peaks, converted_troughs)
        peak_means = [np.mean(vpt[i,max(cp-mr,0):min(cp+mr,vpt.shape[1])]) for cp in converted_peaks]
        trough_means = [np.mean(vpt[i,max(ct-mr,0):min(ct+mr,vpt.shape[1])]) for ct in converted_troughs]

        print(np.mean(vpt[i, :]), trough_means)

def test(runs_path, filter_by):
    sigma = 25  # smoothing level
    mr = 30
    out = load_runs_as_numpy(runs_path,  
                             ['Vaccine prediction total', 'GenLoss', 'Gen Entropy'],
                             filter_by=filter_by)
    vpt = out['Vaccine prediction total']
    gloss = out['GenLoss']
    entropy = out['Gen Entropy']

    gloss_norms = np.linalg.norm(gloss, axis=1, keepdims=True)
    normalized_gloss = gloss / gloss_norms

    entropy_norms = np.linalg.norm(entropy, axis=1, keepdims=True)
    normalized_entropy = entropy / entropy_norms
    summed_arr = np.zeros(vpt.shape)
    for i, y in enumerate(normalized_entropy):
        for j in range(len(y)):
            summed_arr[i,j] = y[j] #normalized_gloss[i, j*2] + y[j]

        smoothed = gaussian_filter1d(summed_arr[i,:], sigma=sigma)
        mins = [np.argmin(smoothed)]

        trough_means = [np.mean(vpt[i,max(ct-mr,0):min(ct+mr,vpt.shape[1])]) for ct in mins]
        print(trough_means)
        print("")

    #idea 1 add gloss to entropy and find the point at which they are the lowest

def find_trend_plateau(y, smooth_sigma=20, threshold=1e-4):
    """
    Detects where a decreasing curve stops declining and starts plateauing or rising.
    
    Parameters:
        y (np.ndarray): 1D signal (e.g., loss curve).
        smooth_sigma (float): Smoothing factor for Gaussian smoothing.
        threshold (float): Derivative threshold to detect plateau or increase.

    Returns:
        (int, float): Index and original value at the turning/plateau point.
    """
    y_smooth = gaussian_filter1d(y, sigma=smooth_sigma)

    dy = np.gradient(y_smooth)

    for i in range(1, len(dy)):
        if dy[i] > threshold and dy[i-1] <= threshold:
            return i, y[i], y_smooth  # return index in original array

    # Fallback: return global min if no trend reversal found
    min_idx = np.argmin(y_smooth)

    return min_idx, y[min_idx], y_smooth

def compute_rolling_std_numpy(y, window=20):
    if window < 1 or window > len(y):
        raise ValueError("Invalid window size")

    pad = window // 2
    padded = np.pad(y, pad_width=pad, mode='reflect')  # or 'edge'

    rolling_std = np.empty(len(y))
    for i in range(len(y)):
        window_slice = padded[i:i + window]
        rolling_std[i] = np.std(window_slice)

    return rolling_std

def normalize_0_1(arr):
    return (arr - np.min(arr)) / (np.max(arr) - np.min(arr))

def entropy_pred_relationship(runs_path, filter_by, num_runs):
    out, event_files = load_runs_as_numpy(runs_path,  
                             ['target prediction', 'GenLoss', 'Gen Entropy', 
                              'tvd', 'Warmup', 'Bias score', 'Gradient Penalty'],
                             filter_by=filter_by,
                             num_runs=num_runs)
    vpt = out['target prediction']
    gloss = out['GenLoss']
    entropy = out['Gen Entropy']
    warmup = out['Warmup']
    bscore = out['Bias score']
    gp = out['Gradient Penalty']
    indices_in_range = np.where((entropy[0] >= 0.75) & (entropy[0] <= 0.8))[0]
    print(indices_in_range)
    #t_f_scores = out['Truth - Fake scores']
    tvd = out['tvd']
    fig, axs = plt.subplots(3, 3, figsize=(10, 10))
    order = np.arange(vpt.shape[0])
    start_point = 0
    print(vpt.shape)
    end_point = len(entropy[0])
    indices = np.arange(len(entropy[0,start_point:end_point]))
    labels = ['entropy', 'truth-fake', 'tvd']
    for i in order[0:9]:
        name = ""
        ''' puts the week/file as the name
        parts = event_files[i].split("||", 2)
        if len(parts) >= 3:
            name = parts[1]
        if i > vpt.shape[0]:
            break
        '''
        #warmup_mean = np.mean(warmup[i,-50:])
        name = "" #str(round(warmup_mean,3))
        
        row, col = divmod(i, 3)
        minidx, _, smoothed = find_trend_plateau(vpt[i], smooth_sigma = 20)
        print(minidx, np.mean(vpt[i,minidx-2:minidx+2]))
        norm_entrop = normalize_0_1(entropy[i,start_point:end_point])
        norm_tvd = normalize_0_1(tvd[i,start_point:end_point])
        norm_score = normalize_0_1(bscore[i,start_point:end_point])
        norm_gp = normalize_0_1(gp[i,start_point:end_point])
        x_axis =  entropy[i,start_point:end_point]
        y_axis = vpt[i,start_point:end_point]
        sc = axs[row,col].scatter(x_axis,y_axis,c=indices, cmap='viridis')
        axs[row,col].set_title(name)
        #sc = axs[row+1,col].scatter(t_f_scores[i,start_point:end_point],vpt[i,start_point:end_point],c=indices, cmap='viridis')
        #sc = axs[row+2,col].scatter(tvd[i,start_point:end_point],vpt[i,start_point:end_point],c=indices, cmap='viridis')
        #fig.text(0.05, 0.85 - i * 0.3, labels[i], va='center', ha='right', fontsize=12)

    if sc is not None:
        cbar = fig.colorbar(sc, ax=axs, orientation='vertical', shrink=0.8)
        cbar.set_label('Time Index')

    return indices_in_range

    #data = np.load("./saves/data.npz")
    #weights = np.load("./saves/weight_history.npz")

    #labels = data['y']
    #weights = weights['w']
    #pws = (weights @ labels).T
    #print("diff: ", np.linalg.norm(pws-vpt))

def success_rate_by_window_comparison(runs_path, filter_by, num_runs, window_length):
    """
    Compares the average of the first K and last K points for each row in a 2D array.

    Parameters:
    - loss_array: np.ndarray of shape (num_runs, time_steps)
    - K: int, number of points to average at the start and end

    Returns:
    - success_percent: float, percentage of rows where avg(first K) > avg(last K)
    """
    out = load_runs_as_numpy(runs_path,  
                             ['tvd', 'GenLoss', 'Gen Entropy'],
                             filter_by=filter_by,
                             num_runs=num_runs)
    loss_array = out['tvd']
    if window_length <= 0 or window_length > loss_array.shape[1] // 2:
        raise ValueError("K must be > 0 and <= half the number of time steps per row")

    start_avg = np.mean(loss_array[:, :window_length], axis=1)
    end_avg = np.mean(loss_array[:, -window_length:], axis=1)
    success_mask = start_avg > end_avg
    success_percent = 100 * np.mean(success_mask)

    return success_percent

def success_rate_JSD_loss(runs_path, filter_by, num_runs, window_length,override_loss_array=None):
    if override_loss_array is None:
        out = load_runs_as_numpy(runs_path,  
                                ['tvd', 'GenLoss', 'Gen Entropy'],
                                filter_by=filter_by,
                                num_runs=num_runs)
        loss_array = out['tvd']
    else:
        loss_array = override_loss_array
    nearly_identical = return_success_proportions(loss_array, 
                        0.03,
                        window_length)
    excellent = return_success_proportions(loss_array, 
                        0.05,
                        window_length)
    very_similar = return_success_proportions(loss_array, 
                        0.07,
                        window_length)
    print("Nearly Identical (0.03): ", nearly_identical)
    print("")
    print("Excellent (0.05): ", excellent)
    print("")
    print("Very similar (0.07): ", very_similar)
    return loss_array

def return_success_proportions(loss_array, 
                               threshold,
                               window_length):
    end_avg = np.mean(loss_array[:, -window_length:], axis=1)
    success_mask = end_avg <= threshold
    success_percent = 100 * np.mean(success_mask)
    mask = np.any(loss_array < threshold, axis=1)
    # Calculate percentage
    percent = 100 * np.sum(mask) / loss_array.shape[0]
    return success_percent, percent

def create_joint_table(runs_path, 
                       threshold,
                       filter_by,
                       var_list):
    out, names = load_all_as_dict(runs_path,  
                                ['tvd', 'GenLoss', 'Gen Entropy'],
                                filter_by=filter_by,)
    loss_array = out['tvd']

    results_dict = {}
    for name, loss in zip(names, loss_array):
        #determine what vars are in the name
        print(loss[0])
        var_count = 0
        c_vars = []
        for v in var_list:
            if v in name:
                var_count += 1
                c_vars.append(v)
        if len(c_vars) > 2:
            print(c_vars)
            print("ERROR: should only be 2 variables")
            assert 1 == 0
        c_vars = sorted(c_vars)

        #create results key
        new_key = tuple(c_vars)
        new_result = int(np.any(loss < threshold, axis=0))

        #store result in list
        if new_key not in results_dict:
            results_dict[new_key] = []
        results_dict[new_key].append(new_result)
    data = results_dict
    unique_strings = sorted(set([s for pair in data.keys() for s in pair]))

    # Create an empty DataFrame
    table = pd.DataFrame(index=unique_strings, columns=unique_strings, dtype=float)

    # Fill in the DataFrame with averages
    for (x, y), values in data.items():
        avg = np.mean(values)
        table.loc[x, y] = avg

    print(table)
    return results_dict, table

if False:
    #file_paths = ["./runs_allweeks/"]
    file_paths = ["./runs/"] 

    for runs_path in file_paths:
        indices = entropy_pred_relationship(runs_path,
                                filter_by=["Week=ALL"], 
                                    num_runs=20,)

        '''
        success_rate = success_rate_by_window_comparison(runs_path,
                                                        filter_by="", 
                                                        num_runs=20,
                                                        window_length=30)
        print(success_rate)
        var_names = ['AGE', 'SEX', 'REGION', 'EDUC', 'MARST', 'RACE', 'INCTOT']
        pair_list = list(combinations(var_names, 2))
        for var in pair_list:
            print(str(var) + " results: ")
            jsd_succ = success_rate_JSD_loss(runs_path, 
                                filter_by=list(var),
                                num_runs=300,
                                window_length = 10)
            print("")
        
        var_names = ['AGE', 'SEX', 'REGION', 'EDUC', 'MARST', 'RACE', 'INCTOT']
        results_dict, table = create_joint_table(runs_path, 
                                                0.07,
                                                filter_by = "",
                                                var_list = var_names)
        '''

In [ ]:
to_load = [str(i) for i in range(23,30)]
to_load = to_load + ["ALL"]
runs_path = "./runs/"
all_outs = {}
for key in to_load:
    out, _ = load_runs_as_numpy(runs_path,  
                             ['target prediction', 'GenLoss', 'Gen Entropy', 
                              'tvd', 'Warmup', 'Bias score', 'Gradient Penalty'],
                             filter_by=["Week="+str(key)],
                             num_runs=20)
    all_outs[key] = out

In [ ]:
for key in all_outs:
   if key == 'ALL':
      continue
   entropy = all_outs[key]['Gen Entropy']
   vpt = all_outs[key]['target prediction']
   all_min_entropy_idx = np.argmin(entropy,axis=1)
   all_min_entropy_preds = []
   for i, amei in enumerate(all_min_entropy_idx):
      all_min_entropy_preds.append(vpt[i,amei])
   all_min_entropy_preds = np.array(all_min_entropy_preds)
   sorted_min_entropy_preds = np.sort(all_min_entropy_preds)

   #isolate out the median and which trial it came from
   median = sorted_min_entropy_preds[(sorted_min_entropy_preds.shape[0]-1)//2]
   selected_trial = np.where(all_min_entropy_preds == median)[0].item()


   #load in the weights and pick out the corresponding weight
   dirr = './saves_to_share_V3/saves_week'+key
   data = np.load(dirr+"/data_"+str(selected_trial)+".npz")
   weights = np.load(dirr+"/weight_history_"+str(selected_trial)+".npz")
   datum = data['x']
   labels = data['y']
   weights = weights['w']
   pws = ((weights @ labels).T.flatten())
   #print(datum.shape, labels.shape)
   closest_time_step = np.where(np.isclose(pws,median))[0].item()
   selected_weights = weights[closest_time_step,:]
   #np.savez(dirr+"/chosen_weight.npz", w=selected_weights)
   #np.savez(dirr+"/chosen_data.npz", x = datum, y = labels)
   #np.savez(dirr+"/chosen_weight_history.npz", w = weights)
   print(key, median)

In [ ]:
for key in range(23,30):
    dirr = "./saves_to_share_V2/saves_week"+str(key)
    data = np.load(dirr+"/chosen_data.npz")
    weights = np.load(dirr+"/chosen_weight.npz")

    data_loaded = data['x']
    labels_loaded = data['y']
    weights_loaded = weights['w']

    print(np.expand_dims(weights_loaded,0) @ labels_loaded)

In [ ]:
 dirr = 'saves_week'+key    
for run_id in range(10):
        data = np.load("./"+dirr+"/data_0.npz")
        weights = np.load("./"+dirr+"/weight_history_0.npz")

    labels = data['y']
    weights = weights['w']
    pws = ((weights @ labels).T.flatten())
    min_index = np.argmin(pws[-20:])
    print(pws[-20:])
    np.savez("./"+dirr+"/chosen_weight.npz", w=weights[indices[min_index]])

In [ ]:
#computing average vaccination
from tensorboard.backend.event_processing import event_accumulator
import os
# Path to the directory where SummaryWriter saved logs
runs_path = "./runs_aggregate_noLRSchedule/runs_week29/"

run_folders = []
for name in os.listdir(runs_path):
    if os.path.isdir(os.path.join(runs_path,name)):
        run_folders.append(os.path.join(runs_path,name))

event_files = []
for run_folder in run_folders:
    for name in os.listdir(run_folder):
        if os.path.isdir(os.path.join(run_folder,name)):
            continue
        else:
            event_files.append(os.path.join(run_folder,name))

list_of_ending_predictions = []
distances = []
for i, event_file in enumerate(event_files):
    # Load event accumulator
    ea = event_accumulator.EventAccumulator(event_file)
    ea.Reload()

    # List available tags (scalars, histograms, images, etc.)
    # Read scalar values (e.g., 'loss', 'accuracy')
    prediction_events = ea.Scalars("Vaccine prediction")
    l2_events = ea.Scalars("L2 Demographics")
    #summ = 0
    #for iter, event in enumerate(prediction_events):
    #    print(event, l2_events[iter])
    list_of_ending_predictions.append(np.mean(prediction_events[-1].value))
    distances.append(l2_events[-1].value)

weights = 1 / (np.array(distances) + 1e-10)
weights /= weights.sum()
print(np.average(list_of_ending_predictions,weights=weights))
print(np.average(list_of_ending_predictions), np.std(list_of_ending_predictions))

In [ ]:
#creating bias xbox and gt
GT_SIZE = 10000
BIAS_SIZE = 5000

test = XBoxDatasetSimulation("./data/gcHouse,7attributes.csv")
def save_new_XBOX_csvs():
    global_GT_dict, global_GT_var_order,global_bias_dict,global_bias_var_order = XBOX_get_GT_and_bias_ratios()

    GT_persons_count = get_all_persons_types_count(GT_SIZE,global_GT_dict)
    BT_persons_count = get_all_persons_types_count(BIAS_SIZE,global_bias_dict)

    GT_sampled_df = XBOX_get_sampled_df(global_GT_var_order,
                                        GT_persons_count,
                                        test.df)
    bias_sampled_df = XBOX_get_sampled_df(global_bias_var_order,
                                        BT_persons_count,
                                        test.df)

    bias_df_normalized, bias_NaN_columns = normalize_df(bias_sampled_df)
    gt_df_normalized, gt_df_NaN_columns = normalize_df(GT_sampled_df)
    all_NaNs_columns = bias_NaN_columns and gt_df_NaN_columns

    clean_NaN_by_col_index(bias_df_normalized,all_NaNs_columns)
    clean_NaN_by_col_index(gt_df_normalized,all_NaNs_columns)

    print(bias_df_normalized.shape, gt_df_normalized.shape)
    #save to CSV files
    bias_df_normalized = bias_df_normalized.sample(frac=1).reset_index(drop=True)
    gt_df_normalized = gt_df_normalized.sample(frac=1).reset_index(drop=True)

    bias_df_normalized.to_csv(BIAS_SAVE_PATH + "XBOX_bias.csv",index=False)
    gt_df_normalized.to_csv(GT_SAVE_PATH + "XBOX_GT.csv",index=False)

save_new_XBOX_csvs()

In [ ]:
#data analysis:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

def analyze_dataset_differences(A, B, feature_names=None):
    """
    Analyzes differences between two datasets A and B.

    Args:
        A (np.ndarray): Dataset A, shape (N, D)
        B (np.ndarray): Dataset B, shape (M, D)
        feature_names (list[str], optional): List of feature names for plotting

    Returns:
        dict: Dictionary of comparison metrics per feature
    """
    assert A.shape[1] == B.shape[1], "Datasets must have the same number of features"
    D = A.shape[1]
    results = {}

    if feature_names is None:
        feature_names = [f"Feature {i}" for i in range(D)]

    for i in range(D):
        feat_A = A[:, i]
        feat_B = B[:, i]
        
        # Basic statistics
        mean_A, std_A = np.mean(feat_A), np.std(feat_A)
        mean_B, std_B = np.mean(feat_B), np.std(feat_B)
        
        # KS-test for distributional difference
        ks_stat, ks_pval = ks_2samp(feat_A, feat_B)

        results[feature_names[i]] = {
            "mean_A": mean_A,
            "mean_B": mean_B,
            "std_A": std_A,
            "std_B": std_B,
            "ks_stat": ks_stat,
            "ks_pval": ks_pval,
        }

        # Plot distributions
        plt.figure(figsize=(6, 4))
        sns.kdeplot(feat_A, label='Dataset A', fill=True)
        sns.kdeplot(feat_B, label='Dataset B', fill=True)
        plt.title(f"Distribution of {feature_names[i]}")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return results
GT_LIMIT = 25000
BIAS_LIMIT = 1000
seed=0
rngs = set_seed(seed,
                device,
                data_init=[True,True],
                data_gen =True,
                network_init=True,)
week='29'
D1_rngs = copy.deepcopy(rngs)
D1_rngs['seed_bias'] = 359556
D2_rngs = copy.deepcopy(rngs)
D2_rngs['seed_bias'] = 280651
#load in both data sets
D1 = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                            bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                            rngs=D1_rngs,
                            device=device,
                            gt_limit = GT_LIMIT,
                            bias_limit = BIAS_LIMIT, 
                            )
D2 = HouseholdPulse_dataset(ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv',
                            bias_path = './data/censusHouseholdPulse_data/pulse_week'+week+'_cleaned.csv',
                            rngs=D2_rngs,
                            device=device,
                            gt_limit = GT_LIMIT,
                            bias_limit = BIAS_LIMIT, 
                            )

analyze_dataset_differences(D1.unscaled_biased, D2.unscaled_biased, feature_names=None)

In [ ]:
#deepset classifier helper functions
def sample_dataset(dataset, batch_size, sample_size, rngs, weights=None):
    if weights is None:
        index = np.random.choice(np.arange(len(dataset)), size=sample_size*batch_size, p=weights)
    else:
        index = rngs['np'].choice(np.arange(len(dataset)),size=sample_size*batch_size)
    sampled_data = torch.clone(dataset[index])
    sampled_data = torch.reshape(sampled_data, (batch_size, sample_size, dataset.shape[1]))
    return sampled_data

def binary_accuracy_from_logits(logits, targets, threshold=0.5):
    preds = (logits > threshold).float()
    correct = (preds == targets).sum().item()
    total = targets.size(0)
    return 100.0 * correct / total


In [ ]:
#making new household census data

from HouseholdCensusDataProcessing import * 
census_df = None
survey_df = None

for w in range(29,30):
    print("processing: ", w)
    week = str(w)
    #census_df = pd.read_csv("./data/censusHouseholdPulse_data/usa_00008.csv")
    survey_df = pd.read_csv("./data/censusHouseholdPulse_data/raw_survey/pulse2021_puf_"+week+".csv")
    survey_df, census_df = recoding_survey_and_census_data(survey_df, census_df, target_var=['RECVDVACC'])
    survey_df.to_csv('./data/censusHouseholdPulse_data/cleaned/pulse_week'+week+'_cleaned.csv', index=False)
    #census_df.to_csv('./data/censusHouseholdPulse_data/cleaned/ipums_cleaned.csv',index=False)

In [ ]:
#compare dataframes
def compare_dataframes(df1, df2):
    assert list(df1.columns) == list(df2.columns), "Columns must match"

    cols = df1.columns
    n_cols = len(cols)
    n_rows = (n_cols + 2) // 3  # auto-layout: 3 columns per row

    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(cols):
        ax = axes[i]
        ax.boxplot([df1[col].dropna(), df2[col].dropna()], labels=["Survey", "Census"])
        ax.set_title(f"Column: {col}")
        ax.grid(True)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

compare_dataframes(survey_df.iloc[:, 1:],census_df.iloc[:,1:])

In [ ]:
def sample_categorical_distribution(df, column, target_dist, K, replace=False, random_state=None):
    """
    Sample K rows from df such that the distribution of the values in the specified column
    matches the target categorical distribution.

    Parameters:
        df (pd.DataFrame): Original DataFrame.
        column_index (int): Index of the column to match distribution on.
        target_dist (dict): Target distribution (e.g., {0: 0.5, 1: 0.3, 2: 0.2}).
        K (int): Total number of samples to draw.
        replace (bool): Whether to sample with replacement.
        random_state (int or None): Seed for reproducibility.

    Returns:
        pd.DataFrame: Sampled DataFrame of size K.
    """
    np.random.seed(random_state)
    result_dfs = []
    
    for category, proportion in target_dist.items():
        num_samples = int(round(proportion * K))
        subset = df[df[column] == category]
        
        if len(subset) == 0:
            raise ValueError(f"No samples found for category '{category}' in the specified column.")
        if not replace and num_samples > len(subset):
            raise ValueError(f"Not enough samples in category '{category}' to sample {num_samples} without replacement.")
        
        sampled = subset.sample(n=num_samples, replace=replace, random_state=random_state)
        result_dfs.append(sampled)
    
    result = pd.concat(result_dfs).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # Fix any rounding issues (e.g., if total != K due to rounding)
    if len(result) > K:
        result = result.sample(n=K, random_state=random_state)
    elif len(result) < K:
        extra = df.sample(n=K - len(result), replace=replace, random_state=random_state)
        result = pd.concat([result, extra]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    return result

def solo_var_experiment(ground_truth_path,
                        target_column,
                        target_dist,
                        gt_size,
                        bias_size,):
    raw_gt_load = pd.read_csv(ground_truth_path)

    biased_dataset = sample_categorical_distribution(raw_gt_load, target_column, target_dist, bias_size, replace=False, random_state=None).to_numpy(dtype=np.float, na_value=0)
    gt_dataset = raw_gt_load.sample(n=gt_size).to_numpy(dtype=np.float, na_value=0)
    

ground_truth_path='./data/censusHouseholdPulse_data/ipums_cleaned.csv'

In [ ]:
#entropy experiments
NUM_POINTS = 5000
NUM_FOCUS= 100
NUM_REST = NUM_POINTS - NUM_FOCUS

concentrated = np.ones(NUM_FOCUS,) * (NUM_REST*3/NUM_FOCUS)
rest = np.ones(NUM_REST)

total_pr = torch.from_numpy(np.concatenate((concentrated, rest)) / (sum(concentrated) + + sum(rest)))

entropy = -torch.sum(total_pr * torch.log(total_pr), dim=0)
max_entropy = torch.log(torch.tensor(NUM_POINTS, dtype=entropy.dtype, device=entropy.device))

In [ ]:
%autoreload 2
#making new ari data set
from AriSurveyDataProcessing import * 
census_df = None
survey_df = None

census_df = pd.read_csv("./data/ari_survey/usa_00009.csv")
survey_df = pd.read_csv("./data/ari_survey/aug23data.csv")
#survey_df, census_df = recoding_survey_and_census_data(survey_df, census_df)
#survey_df.to_csv('./data/ari_survey/aricleaned.csv', index=False)
#census_df.to_csv('./data/ari_survey/ipums_cleaned.csv',index=False)

In [ ]:
survey_df['age']

In [ ]:
accepted_answers = ['Full-time', 'Retired', 'Part-time', 'Unemployed', 'Homemaker', 'Permanently disabled',
                            'Student', 'Other, please specify:', 'Temporarily laid off']
survey_df[survey_df['employmentstatus'].isin(accepted_answers)]

Transform raw data table into discretized/binned data 
1. Remove irrelevant columns
    a. List out available columns - "list(df.columns)"
    b. pick out variables that have an IPUMS equivalent
    c. Remove irrelevant columns - df = df[relevant_columns] (relevant_columns: list) 
    d. or call df = df.filter(items=relevant_survey_col)
2. "fix" (remove) NaN entries in relevant columns
    a. check responses - df[var_name].value_counts()
    b. create a list of "accepted_answers"
    c. use accepted answers to create a "mask" - df[var_name].isin(accepted_answers)
    d. use mask to remove bad columns - df = df[bad_answer_mask]
    e. Next - check for na entries. create a mask - df[var_name].isna()
    f. remove NaN entries df = df[na_mask]
3. Bin relevant columns
    a. create local function that takes "value" as input and returns the appropriate bin
    b. call df[var_name].apply(binning_function)
    c. Store df's as CSV files
4. Additional but important modifcations for the NN. 
    a. The features/variables for survey/census appear in the same order. 
        - in (1) when you create list of relevant columns. calling df[relevant_columns], will automatically reorder into the same order
    b. What happens if variable names are not the same? e.g. age vs AGE
        - df.rename(columns={old_name:new_name},inplace=True)
        - I renamed all the survey columns to the census name

Now we have the data set made

How to call the actual function - creating the data set object
1. One column must exist in a specific location in the survey data set - column 0 must be the labels
2. one column must exist in a specific location in the census data set - column 0 must be the PERWT IPUMS variable
3. Create class new_dataset(HouseholdPulse_dataset) that inherits household pulse object. 
    a. This object takes:
        - file path - str - to survey dataframe
        - file path - str - to census data frame
        - GT_LIMIT - int - number of census points to use
        - BIAS_LIMIT - int - number of survey points to use

How to call the actual function - GAN object
1. create a WGAN_GP object that takes as input:
    a. data set (created above) object
    b. Tuning parameters:
        - generator_type='deepSet' - generator architure. default - deepSet; no real need to change
        - discriminator_type='deepSet' - critic/discriminator architure. default - deepSet; no real need to change
        - gen_learning_rate=hparams["glearningrate"] - float - controls how fast generator converges. might need tuning
        - disc_learning_rate=hparams["dlearningrate"] - float - controls how fast disc converges. might need tuning 
        - batch_size=hparams["batch_size"] - int - can keep at default (16), how many "subsets" of data points the discriminator sees at a time
        - subset_size - int - default (128) how many data points on which the discriminator makes a decision. 
        - gen_layers=hparams["gen_layers"] - list[int] - size of the gen network, default size is 1 hidden layer of 1024. This will have minor impact
        - disc_layers=hparams["disc_layers"] - list[int] - size of the disc network, default size is 1 hidden layer of 1024
        - lambda_gp=hparams["lambdagp"] - float - default - keep for theoretical guaruntees 
        - lambda_weights=hparams["lambdaw"] - float - default - punishes high concentration of probabilities into a small number of points
        - lambda_demo=hparams["lambdad"] - float - default - punishes high deviation from census demographic 
        - temperature=hparams["tau"] - float - default - needed for generator
        - generator_dropout=hparams["generator_dropout"] - float - default - prevents overfitting
        - discriminator_dropout=hparams["discriminator_dropout"] - float - default - prevents overfitting

Measuring demographic similarity between a weighted survey and a census data set:
1. Currently - treat each variable independently
    a. For each variable, measure its JSD similarity to the census (histogram comparison)
    b. average the histogram similarity score
    c. similar to raking - treat variables independently, how closely can i match each of the distributions
2. Alternative - treat all variables simultaneously using the intersection of individuals 
    a. caveat - whent here are a lot of variables, the intersected space gets extremely large (e.g. 5 bins with 9 variables, thats 5^9 unique individuals)
        - there are a lot of combinations in the census that wont exist in the survey, and vice versa
    b. multi-level modeling post stratification - does groups of variables at a time, rather than the full span of variables
    c. Even if we trained each var independently, does it correct for the skew of specific interactions
    d. Synthetic experiments
        - Upscale a specific interaction of 2 - 3 variables (start with 2)
        - Train system using default method - use all the variables (10 variables)
        - Meausre the recovery (or lack thereof) of the upscaled interaction - measuring the balance of all interactions of the 2-3 variables

In [ ]:
educ_mapping = {
            'Less than high school': 1,  # N/A or no schooling
            'Some high school': 1,
            'High school graduate or equivalent (for example GED)': 2,  # High school graduate
            'Some college, but degree not received or is in progress': 3,  # Some college
            'Associate\'s degree (for example AA, AS)': 3,  # Some college
            'Bachelor\'s degree (for example BA, BS, AB)': 4, # Bachelor's degree
            'Graduate degree (for example master\'s, professional, doctorate)': 5  # Graduate degree
        }

survey_df['education'] = survey_df['education'].map(educ_mapping)
survey_df['education'].value_counts()

In [ ]:
survey_df['education'].value_counts()

In [ ]:
census_df['AGE'].value_counts()

In [ ]:
survey_df_orig['employmentstatus'].value_counts()